# Lab 6 — Build the course assistant

**Artificial Intelligence · 4th year · UIR · Pr. Hakim Hafidi**

| | |
|---|---|
| **Duration** | 2 hours, individual work |
| **Runtime** | CPU is enough — but you need your `GEMINI_API_KEY` from Labs 1–2 |
| **You need** | the four study sheets (`Session_1.pdf` … `Session_4.pdf`) and your `lab05_<id>.json` |
| **You will learn** | how to carry provenance through an index · how to make a model answer from passages and cite them · how to handle follow-up questions · how to test an assistant the way you would test any deliverable |
| **Deliverable** | this notebook, executed, + `lab06_<your id>.json`, both on connect.uir.ac.ma |
| **Grading** | 4 points: it runs (1) · the acceptance test is complete and correct (1) · your diagnosis of a failure (1) · the release note (1) |

Last week you built a retriever and measured it. Today you put a language model on the other
end of it and ship **the course assistant**: a thing that answers questions about Sessions 1
to 4, says where each claim comes from, and admits when it does not know.

At the end of the session you will have a link you can send to someone.

> **Four functions to write today**, each from a written contract, each checked by tests you
> must make pass. Then the part that is not code: deciding whether what you built is good
> enough to hand over, and saying so honestly.

> **One rule for the whole lab: `temperature = 0`.** Otherwise the acceptance test in Part E
> gives you a different result every time you run it, and you cannot tell a fix from noise.

---
## Step 0 — Setup (10 minutes)

Three things to upload: the four PDFs, and your `lab05_<id>.json` from last week.

**If you lost your Lab 5 file**, do not panic — the toolbox falls back to a default
configuration and a default set of questions, and you can do the whole lab. You will just be
working with someone else's questions rather than your own, and you should say so in Part F.

In [2]:
%pip install -q -U sentence-transformers rank-bm25 pypdf google-genai "gradio>=4.44"
print("done — if Colab asks you to restart the session, do it, then re-run this cell.")

done — if Colab asks you to restart the session, do it, then re-run this cell.


In [3]:
import pathlib
from google.colab import files

if len(sorted(pathlib.Path(".").glob("Session_*.pdf"))) < 4:
    print("Select Session_1.pdf … Session_4.pdf (all four at once):")
    files.upload()

if not sorted(pathlib.Path(".").glob("lab05_*.json")):
    print("\nNow select your lab05_<id>.json (or press Cancel if you do not have it):")
    try:
        files.upload()
    except Exception:
        pass

PDFS = sorted(pathlib.Path(".").glob("Session_*.pdf"))
assert len(PDFS) == 4, f"Found {len(PDFS)} PDFs, expected 4: {[p.name for p in PDFS]}"
print("✅", [p.name for p in PDFS])

✅ ['Session_1.pdf', 'Session_2.pdf', 'Session_3.pdf', 'Session_4.pdf']


In [4]:
#@title ⚙️ Lab 6 toolbox — run this cell once, then fold it { display-mode: "form" }
import json, re, time, hashlib, pathlib, unicodedata, textwrap
import numpy as np, pandas as pd

CALL_LOG = []
MODEL_FAST = "gemini-3.5-flash-lite"   # check this name the morning of the lab

# The assistant must produce this sentence, in English, whatever the language of the
# question, when the passages do not contain the answer. Part E detects abstention by
# looking for it, so it has to be exact.
REFUSAL = "I could not find this in the course documents."

# ------------------------------------------------------------------ reading PDFs
def extract_pdf_pages(path):
    """Return ONE STRING PER PAGE, in order. Page numbers are 1-based: page 1 is pages[0].

    Same whitespace normalisation as Lab 5 (runs of spaces and tabs collapsed, line breaks
    kept). The difference with last week is that nothing is joined: you need the page
    boundaries, because a citation that cannot name a page is not a citation."""
    from pypdf import PdfReader
    out = []
    for p in PdfReader(str(path)).pages:
        t = p.extract_text() or ""
        out.append("\n".join(re.sub(r"[ \t\xa0]+", " ", l).strip() for l in t.split("\n")))
    return out

def clean_page(text):
    """Strip the running header and footer. You wrote this in Lab 5 — here it is, solved,
    so you can spend the two hours on what is new."""
    text = re.sub(r"AI\s*·\s*4th year\s*·\s*UIR\s*—\s*Session\s*\d+", " ", text)
    text = re.sub(r"Pr\.\s*Hakim\s*Hafidi", " ", text)
    text = re.sub(r"Artificial Intelligence\s*—\s*study sheet", " ", text)
    text = re.sub(r"page\s*\d+\s*/\s*\d+", " ", text, flags=re.I)
    return re.sub(r"[ \t\xa0]+", " ", text).strip()

# ------------------------------------------------------------------ retrieval (Lab 5)
_models = {}
def get_model(name):
    from sentence_transformers import SentenceTransformer
    if name not in _models:
        print(f"loading {name} …")
        _models[name] = SentenceTransformer(name)
    return _models[name]

EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

def embed(texts, model_name=EMBED_MODEL, batch_size=32):
    """Unit-length vectors, one row per string. A dot product between rows IS the cosine.
    The multilingual model is the default this week: your assistant must answer questions
    asked in French about an English corpus, which is exactly the Part E result of Lab 5."""
    v = get_model(model_name).encode(list(texts), batch_size=batch_size,
                                     show_progress_bar=False, convert_to_numpy=True,
                                     normalize_embeddings=True)
    return v.astype(np.float32)

def tokenize(text):
    text = unicodedata.normalize("NFKD", text.lower())
    text = "".join(c for c in text if not unicodedata.combining(c))
    return re.findall(r"[a-z0-9]+", text)

_INDEX = {"texts": None, "matrix": None, "bm25": None}

def build_index(chunk_texts):
    """Embed every chunk and build the keyword index. Call this once, after Part A."""
    from rank_bm25 import BM25Okapi
    t0 = time.time()
    _INDEX["texts"] = list(chunk_texts)
    _INDEX["matrix"] = embed(chunk_texts)
    _INDEX["bm25"] = BM25Okapi([tokenize(t) for t in chunk_texts])
    print(f"✅ indexed {len(chunk_texts)} chunks in {time.time()-t0:.1f}s")

def hybrid_search(query, top_k=5, K=60):
    """Vector search + BM25, merged by reciprocal rank fusion. Returns a list of indices
    into the chunk list you passed to build_index(), best first.

    This is your Lab 5 work, handed back to you solved so that today is about what comes
    after retrieval. Scores are deliberately not returned: RRF merges RANKS, and a fused
    score means nothing on its own."""
    assert _INDEX["matrix"] is not None, "Call build_index(CHUNK_TEXTS) first."
    n = len(_INDEX["texts"])
    vec_rank = np.argsort(-(_INDEX["matrix"] @ embed([query])[0]))[:50].tolist()
    bm_rank  = np.argsort(-np.array(_INDEX["bm25"].get_scores(tokenize(query))))[:50].tolist()
    fused = {}
    for ranking in (vec_rank, bm_rank):
        for rank, idx in enumerate(ranking, start=1):
            fused[idx] = fused.get(idx, 0.0) + 1.0 / (K + rank)
    return [i for i, _ in sorted(fused.items(), key=lambda kv: -kv[1])][:top_k]

# ------------------------------------------------------------------ the model
_client = None
def _gemini():
    global _client
    if _client is None:
        from google import genai
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        assert key, "No GEMINI_API_KEY in Colab Secrets — see Lab 1, Step 0."
        _client = genai.Client(api_key=key.strip())
    return _client

CACHE = pathlib.Path("llm_cache"); CACHE.mkdir(exist_ok=True)
_last = [0.0]
MIN_INTERVAL = 4.2      # the free tier allows about 15 requests per minute

def call_model(prompt, temperature=0.0, max_tokens=900, retries=4, use_cache=True):
    """Send one prompt, get one answer back, with a disk cache.

    The cache is keyed on (model, prompt, temperature, max_tokens). You are going to re-run
    these cells many times today — without it you would burn your daily quota before Part E.
    A cached call costs nothing and returns instantly; pass use_cache=False to force a real
    call. Returns {"text", "latency_s", "tokens_in", "tokens_out", "cached"}."""
    key = hashlib.sha256(json.dumps([MODEL_FAST, prompt, temperature,
                                     max_tokens]).encode()).hexdigest()[:24]
    f = CACHE / f"{key}.json"
    if use_cache and f.exists():
        d = json.loads(f.read_text()); d["cached"] = True
        CALL_LOG.append({k: d[k] for k in ("latency_s", "tokens_in", "tokens_out", "cached")})
        return d

    from google.genai import types
    delay = 2.0
    for _ in range(retries):
        wait = MIN_INTERVAL - (time.time() - _last[0])
        if wait > 0: time.sleep(wait)
        try:
            t0 = time.time()
            r = _gemini().models.generate_content(
                model=MODEL_FAST, contents=prompt,
                config=types.GenerateContentConfig(temperature=temperature,
                                                   max_output_tokens=max_tokens))
            _last[0] = time.time()
            u = getattr(r, "usage_metadata", None)
            d = {"text": (r.text or "").strip(),
                 "latency_s": round(time.time() - t0, 2),
                 "tokens_in": getattr(u, "prompt_token_count", 0) or 0,
                 "tokens_out": getattr(u, "candidates_token_count", 0) or 0,
                 "cached": False}
            if not d["text"]:
                raise RuntimeError("empty answer — raise max_tokens and try again")
            f.write_text(json.dumps(d))
            CALL_LOG.append({k: d[k] for k in ("latency_s", "tokens_in", "tokens_out", "cached")})
            return d
        except Exception as e:
            if "404" in str(e) or "NOT_FOUND" in str(e):
                raise RuntimeError(f"Model {MODEL_FAST!r} does not exist for this key. "
                                   f"See https://ai.google.dev/gemini-api/docs/models and "
                                   f"update MODEL_FAST in the toolbox. | {e}")
            _last[0] = time.time()
            print(f"  ⏳ {str(e)[:120]} — waiting {delay:.0f}s"); time.sleep(delay); delay *= 2
    raise RuntimeError("four attempts failed — check your key and your quota")

# ------------------------------------------------------------------ Lab 5 carry-over
DEFAULT_SIZE, DEFAULT_OVERLAP = 200, 40

DEFAULT_EVAL = [
 {"q": "Which workshop gave artificial intelligence its name, and in what year?",
  "doc": "Session_1"},
 {"q": "What did the Lighthill report do to AI funding in Britain?", "doc": "Session_1"},
 {"q": "Why could a single perceptron not solve XOR?", "doc": "Session_2"},
 {"q": "What did AlexNet change in 2012?", "doc": "Session_2"},
 {"q": "What is the difference between supervised and unsupervised learning?",
  "doc": "Session_3"},
 {"q": "What is overfitting, and how do you detect it?", "doc": "Session_3"},
 {"q": "How many parameters does GPT-3 have?", "doc": "Session_4"},
 {"q": "Why can an n-gram model not generalise?", "doc": "Session_4"},
 {"q": "Qu'est-ce qu'un plongement lexical (embedding) ?", "doc": "Session_4"},
 {"q": "À quoi sert l'encodage positionnel dans un Transformer ?", "doc": "Session_4"},
]

# Five questions whose answer is NOT in Sessions 1-4. The assistant must refuse all five.
UNANSWERABLE = [
 {"q": "What learning rate is used for the LoRA fine-tuning in Lab 3?",
  "why": "a lab detail, never written in the study sheets"},
 {"q": "How does reciprocal rank fusion combine two rankings?",
  "why": "Session 7 material — the corpus stops at Session 4"},
 {"q": "What is the date of the final exam for this course?",
  "why": "administrative, in no document"},
 {"q": "Quelle est la capitale du Maroc ?",
  "why": "the model knows this perfectly well from pretraining — and must still refuse, "
         "because it is not in the passages. This is the question that tells you whether "
         "your assistant is grounded or merely fluent."},
 {"q": "Which optimiser does the Adam paper compare against?",
  "why": "outside the corpus entirely"},
]

def load_lab5(default_size=DEFAULT_SIZE, default_overlap=DEFAULT_OVERLAP):
    """Read your lab05_<id>.json and return (size, overlap, eval_set).
    Falls back to the defaults, loudly, if the file is not there."""
    hits = sorted(pathlib.Path(".").glob("lab05_*.json"))
    if not hits:
        print("⚠️  No lab05_*.json found — using the default configuration and questions.")
        print("   Say so in Part F: your numbers are not comparable with your classmates'.")
        return default_size, default_overlap, [dict(e) for e in DEFAULT_EVAL]
    d = json.loads(hits[0].read_text()).get("extra", {})
    size = int(d.get("my_size", default_size))
    overlap = int(d.get("my_overlap", default_overlap))
    ev = []
    for e in d.get("eval_set", []):
        q = e.get("q") or e.get("question")
        if q: ev.append({"q": q, "doc": e.get("doc", "")})
    if not ev:
        print("⚠️  Your file has no usable questions — using the default set.")
        ev = [dict(e) for e in DEFAULT_EVAL]
    print(f"✅ {hits[0].name}: size={size}, overlap={overlap}, {len(ev)} questions")
    return size, overlap, ev

# ------------------------------------------------------------------ the interface
def launch_ui(answer_fn, title="Course assistant — Sessions 1 to 4"):
    """Put your answer() behind a chat window and return a public link (valid ~72 h).

    Falls back to a plain console loop if gradio misbehaves in your runtime — the lab does
    not depend on the interface working, only on your answer() working."""
    history = []
    def respond(message, chat):
        r = answer_fn(message, history=list(history))
        history.append((message, r["text"]))
        src = ("\n\n*sources: " + ", ".join(r["sources"]) + "*") if r["sources"] else ""
        return r["text"] + src
    try:
        import gradio as gr
        with gr.Blocks(title=title) as demo:
            gr.Markdown(f"## {title}\nAsk in English or in French. "
                        f"Every claim should carry a `[n]` pointing at a passage.")
            try:    bot = gr.Chatbot(height=420, type="messages")
            except TypeError:
                    bot = gr.Chatbot(height=420)
            box = gr.Textbox(placeholder="Your question…", show_label=False)
            clear = gr.Button("Clear conversation")
            def _send(msg, chat):
                chat = chat or []
                out = respond(msg, chat)
                if chat and isinstance(chat[0], dict):
                    chat = chat + [{"role": "user", "content": msg},
                                   {"role": "assistant", "content": out}]
                else:
                    try:    chat = chat + [{"role": "user", "content": msg},
                                           {"role": "assistant", "content": out}]
                    except Exception: chat = chat + [(msg, out)]
                return chat, ""
            box.submit(_send, [box, bot], [bot, box])
            clear.click(lambda: ([], history.clear())[0], None, bot)
        return demo.launch(share=True, debug=False, quiet=True)
    except Exception as e:
        print(f"⚠️  gradio failed ({str(e)[:110]}) — falling back to the console.")
        print("Type your questions; an empty line stops.\n")
        while True:
            q = input("you > ").strip()
            if not q: break
            print("\nassistant >", respond(q, None), "\n")

def console_ui(answer_fn):
    """The fallback, callable on its own if you prefer it."""
    history = []
    while True:
        q = input("you > ").strip()
        if not q: break
        r = answer_fn(q, history=list(history))
        history.append((q, r["text"]))
        print("\nassistant >", r["text"])
        if r["sources"]: print("sources  >", ", ".join(r["sources"]))
        print()

# ------------------------------------------------------------------ submission
def export_lab(lab_id, student_id, answers: dict, tables: dict = None, extra: dict = None):
    student_id = str(student_id).strip()
    assert student_id and student_id.lower() not in ("", "xxxxx"), "Set STUDENT_ID."
    payload = {"lab": lab_id, "student_id": student_id,
               "answers": {k: str(v).strip() for k, v in answers.items()},
               "tables": {k: (v.to_dict("records") if hasattr(v, "to_dict") else v)
                          for k, v in (tables or {}).items()},
               "extra": extra or {}, "call_log": CALL_LOG,
               "created": time.strftime("%Y-%m-%d %H:%M")}
    name = f"{lab_id}_{student_id}.json"
    pathlib.Path(name).write_text(json.dumps(payload, indent=2, ensure_ascii=False))
    short = [k for k, v in payload["answers"].items() if len(v) < 40]
    print(f"📦 {name} written ({pathlib.Path(name).stat().st_size} bytes)")
    print("⚠️  too short:", short) if short else print("✅ All answers filled.")
    print(f"\nSubmit the notebook AND {name} on connect.uir.ac.ma")
    return payload

print("✅ Lab 6 toolbox loaded.")
print("   retrieval : extract_pdf_pages, clean_page, build_index, hybrid_search")
print("   model     : call_model (cached, temperature 0 by default), REFUSAL")
print("   lab       : load_lab5, UNANSWERABLE, launch_ui, export_lab")

✅ Lab 6 toolbox loaded.
   retrieval : extract_pdf_pages, clean_page, build_index, hybrid_search
   model     : call_model (cached, temperature 0 by default), REFUSAL
   lab       : load_lab5, UNANSWERABLE, launch_ui, export_lab


In [5]:
STUDENT_ID = "130588"   # TODO: your apogee number
assert STUDENT_ID.strip().upper() != "XXXXX", "Set STUDENT_ID first."

SIZE, OVERLAP, EVAL = load_lab5()
print(f"\nChunking with size={SIZE}, overlap={OVERLAP}")
print(f"{len(EVAL)} questions carried over from Lab 5")

✅ lab05_987.json: size=200, overlap=40, 10 questions

Chunking with size=200, overlap=40
10 questions carried over from Lab 5


---
## Part A — An index that knows where things come from (20 minutes)

In Lab 5 you chunked a wall of text and marked the right answer by a snippet, because chunk
numbers changed every time you re-chunked. That was fine for measuring retrieval. It is not
enough for an assistant: *"three months' notice"* is worth nothing if you cannot say which
document and which page it came from.

So this week every chunk carries its **provenance**. That is the whole of Part A, and it is
the brick the citations of Part B are built on.

In [6]:
pages_by_doc = {p.stem: [clean_page(t) for t in extract_pdf_pages(p)] for p in PDFS}

for name, pages in pages_by_doc.items():
    print(f"{name}: {len(pages)} pages, {sum(len(p.split()) for p in pages):>5} words")

print("\nPage 2 of Session_1, first 300 characters:\n")
print(pages_by_doc["Session_1"][1][:300])

Session_1: 5 pages,  2734 words
Session_2: 5 pages,  2932 words
Session_3: 5 pages,  3159 words
Session_4: 5 pages,  3152 words

Page 2 of Session_1, first 300 characters:

Turing and the universal machine (1936).
In
On Computable Numbers
, Alan Turing described an abstract machine with an infinite tape, a
read-write head, and a table of rules. He proved that a single such machine, given the right description on its tape, can simulate any other. This is
the idea of
sof


In [7]:
# TODO: write `chunk_document()`.
#
#   CONTRACT
#   --------
#   chunk_document(pages, doc_name, size=200, overlap=40) -> list of dicts
#
#   pages     : list of strings, one per page. Page numbers are 1-BASED: pages[0] is page 1.
#   doc_name  : e.g. "Session_3"
#   size      : chunk length, in words
#   overlap   : how many words two consecutive chunks share
#
#   Each dict has EXACTLY these four keys:
#       "text"      the chunk itself, words joined by single spaces
#       "doc"       doc_name
#       "page"      the 1-based page on which the chunk STARTS
#       "chunk_id"  unique across the whole corpus — f"{doc_name}#{i}" is fine
#
#   · chunks are cut over the document's words, in order, with a sliding window
#     (step = size - overlap), exactly as in Lab 5 — but a chunk may now span two pages,
#     and it reports the page where it begins
#   · a document with no words returns []
#   · a document shorter than `size` returns exactly one chunk
#   · overlap >= size is impossible: raise ValueError
#
#   Hint: build two parallel lists first — one word per entry, and the page number of that
#   word — then slide over them. The page of a chunk is the page of its first word.

def chunk_document(pages, doc_name, size=200, overlap=40):
    if overlap >= size or overlap < 0:
        raise ValueError("L'overlap devrait être compris dans [0, size[")

    words, word_page = [], []
    for num_page, texte in enumerate(pages, start=1):
        for w in texte.split():
            words.append(w)
            word_page.append(num_page)

    if not words:
        return []

    step = size - overlap
    chunks, start = [], 0

    while True:
        chunks.append({"text": " ".join(words[start:start + size]),
                       "doc": doc_name,
                       "page": word_page[start],
                       "chunk_id": f"{doc_name}#{len(chunks)}"})
        if start + size >= len(words):
            break
        start += step
    return chunks

assert isinstance(chunk_document(["a b c"], "T"), list), \
    "chunk_document has no body yet — replace the `...` with your implementation."
print("ok — now make the nine tests below pass")

ok — now make the nine tests below pass


In [8]:
# --- the nine tests you must make pass ----------------------------------------
P1 = " ".join(f"p1w{i}" for i in range(100))
P2 = " ".join(f"p2w{i}" for i in range(100))
CS = chunk_document([P1, P2], "Two", size=60, overlap=10)

# T1 — a document with no words produces no chunks
assert chunk_document([], "Empty") == [], "T1: no pages should give no chunks"
assert chunk_document(["", "   "], "Empty") == [], "T1: blank pages should give no chunks"

# T2 — a document shorter than one chunk produces exactly one
one = chunk_document(["alpha beta gamma"], "Tiny", size=200, overlap=40)
assert len(one) == 1, f"T2: expected 1 chunk, got {len(one)}"
assert one[0]["text"] == "alpha beta gamma", "T2: the text must be the whole document"
assert one[0]["page"] == 1 and one[0]["doc"] == "Tiny", "T2: wrong provenance"

# T3 — exactly the four keys, no more, no fewer
assert all(set(c) == {"text", "doc", "page", "chunk_id"} for c in CS), \
    f"T3: wrong keys, got {set(CS[0])}"

# T4 — chunk_id is unique
assert len({c["chunk_id"] for c in CS}) == len(CS), "T4: duplicate chunk_id"

# T5 — pages are 1-based and never go backwards
assert CS[0]["page"] == 1, "T5: the first chunk starts on page 1, not page 0"
assert all(a["page"] <= b["page"] for a, b in zip(CS, CS[1:])), "T5: pages went backwards"

# T6 — a chunk that starts on page 2 says page 2.
# 200 words, size 60, overlap 10 → windows start at words 0, 50, 100, 150.
# Words 0-99 are on page 1, words 100-199 on page 2.
assert [c["page"] for c in CS] == [1, 1, 2, 2], \
    f"T6: expected pages [1, 1, 2, 2], got {[c['page'] for c in CS]}"

# T7 — two consecutive chunks share exactly `overlap` words
a, b = CS[0]["text"].split(), CS[1]["text"].split()
assert a[-10:] == b[:10], "T7: consecutive chunks must share exactly `overlap` words"

# T8 — no word is lost on the way
seen = set()
for c in CS:
    seen.update(c["text"].split())
assert len(seen) == 200, f"T8: {200 - len(seen)} words are in no chunk at all"

# T9 — an impossible configuration is refused, not silently accepted
try:
    chunk_document([P1], "X", size=50, overlap=50)
    raise AssertionError("T9: overlap >= size should raise ValueError")
except ValueError:
    pass

print("✅ nine tests passed")

✅ nine tests passed


In [9]:
CHUNKS = []
for p in PDFS:
    CHUNKS += chunk_document(pages_by_doc[p.stem], p.stem, size=SIZE, overlap=OVERLAP)

CHUNK_TEXTS = [c["text"] for c in CHUNKS]
assert len({c["chunk_id"] for c in CHUNKS}) == len(CHUNKS), "chunk_id is not unique corpus-wide"

print(f"{len(CHUNKS)} chunks from {len(PDFS)} documents\n")
print(pd.DataFrame(CHUNKS).groupby("doc").agg(chunks=("chunk_id", "count"),
                                              last_page=("page", "max")))
print("\nThree chunks, with their provenance:\n")
for c in (CHUNKS[0], CHUNKS[len(CHUNKS)//2], CHUNKS[-1]):
    print(f"  {c['chunk_id']:<14} {c['doc']} p.{c['page']}  {c['text'][:70]}…")

build_index(CHUNK_TEXTS)

76 chunks from 4 documents

           chunks  last_page
doc                         
Session_1      17          5
Session_2      19          5
Session_3      20          5
Session_4      20          5

Three chunks, with their provenance:

  Session_1#0    Session_1 p.1  Session 1 — What is AI, and where does it come from? Artificial Intell…
  Session_3#2    Session_3 p.1  best achieve its goal given what it knows. Note rational , not perfect…
  Session_4#19   Session_4 p.5  the man" equals "the man bit the dog". (c) A pretrained text completer…
loading sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 …


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ indexed 76 chunks in 50.9s


---
## Part B — The assistant (25 minutes)

Three functions, and then you have something that answers.

The one that will cost you time is not the one you expect. `parse_citations` looks trivial
and contains the bug that every RAG system ships at least once: **passages are shown to the
model as `[1]`, `[2]`, `[3]`, and Python lists start at 0.** Get that wrong and your assistant
will cite the right number and name the wrong page — which is worse than citing nothing,
because now it looks trustworthy.

In [10]:
# TODO: write `build_prompt()`.
#
#   CONTRACT
#   --------
#   build_prompt(question, passages, cite=True, refuse=True) -> str
#
#   passages : a list of chunk dicts, in the order retrieval returned them
#
#   The prompt has three parts, in this order: the instructions, the passages, the question.
#
#   · render passage number i (starting at 1) EXACTLY in this shape, one per block:
#         [i] (doc, page N) text of the chunk
#   · the question goes LAST, after every passage
#   · always instruct the model to answer in the same language as the question
#   · if cite=True : instruct it to put the passage number after each claim, like [1].
#     if cite=False: the word "cite" must not appear in the instructions you write
#   · if refuse=True : the prompt contains the constant REFUSAL, character for character,
#     with an instruction to reply with that sentence and nothing else when the passages
#     do not hold the answer
#     if refuse=False: REFUSAL must not appear at all
#   · passages=[] is legal: still return a usable prompt (the model must then refuse)
#
#   REFUSAL is defined in the toolbox. Print it if you want to see it.

def build_prompt(question, passages, cite=True, refuse=True):
    rules = ["You are a course assistant. Answer using ONLY the passages below, never outside knowledge.",
             "Answer in the same language as the question."]
    if cite:
        rules.append("Cite the passage number after each claim, like [1].")
    if refuse:
        rules.append("If the passages do not contain the answer, reply with exactly this sentence "
                     f"and nothing else, in English even if the question is in French: {REFUSAL}")
    blocks = [f"[{i}] ({p['doc']}, page {p['page']}) {p['text']}"
              for i, p in enumerate(passages, start=1)]
    ctx = "\n\n".join(blocks) if blocks else "(no passages were retrieved)"
    return "\n".join(rules) + "\n\nPASSAGES:\n" + ctx + "\n\nQUESTION: " + question

assert isinstance(build_prompt("q", []), str), \
    "build_prompt has no body yet — replace the `...` with your implementation."
print("ok")

ok


In [11]:
# --- seven tests. The synthetic passages below contain no citation-like text on purpose.
PS = [{"text": "Rome and Paris are both capitals.", "doc": "Session_1", "page": 2,
       "chunk_id": "a"},
      {"text": "GPT-3 has 175 billion parameters.", "doc": "Session_4", "page": 5,
       "chunk_id": "b"}]
p = build_prompt("How many parameters does GPT-3 have?", PS)

# T1 — a string that contains the question
assert isinstance(p, str), "T1: build_prompt must return a string"
assert "How many parameters does GPT-3 have?" in p, "T1: the question is missing"

# T2 — every passage appears, with its source
for x in PS:
    assert x["text"] in p, f"T2: passage missing: {x['text']!r}"
assert "Session_4" in p and "page 5" in p, "T2: the provenance is not in the prompt"

# T3 — numbering follows the order given, and stops at n
assert "[1] (" in p and "[2] (" in p, "T3: passages must be numbered [1], [2], …"
assert "[3] (" not in p, "T3: there is no third passage"
assert p.index(PS[0]["text"]) < p.index(PS[1]["text"]), "T3: the order was not preserved"

# T4 — the question comes after every passage
assert p.index("How many parameters") > p.index(PS[1]["text"]), "T4: the question is not last"

# T5 — the refusal sentence, verbatim, and only when asked for
assert REFUSAL in p, "T5: REFUSAL must appear character for character"
assert REFUSAL not in build_prompt("q", PS, refuse=False), "T5: refuse=False still refuses"

# T6 — cite=False removes the instruction to cite
assert "cite" in build_prompt("q", PS, cite=True).lower(), "T6: cite=True must ask for it"
assert "cite" not in build_prompt("q", PS, cite=False).lower(), "T6: cite=False must not"

# T7 — no passages at all is still a usable prompt
empty = build_prompt("anything", [])
assert isinstance(empty, str) and REFUSAL in empty, "T7: an empty prompt must instruct refusal"
assert "[1] (" not in empty, "T7: there is no passage 1 to show"

print("✅ seven tests passed\n")
print(build_prompt("How many parameters does GPT-3 have?", PS)[:700], "…")

✅ seven tests passed

You are a course assistant. Answer using ONLY the passages below, never outside knowledge.
Answer in the same language as the question.
Cite the passage number after each claim, like [1].
If the passages do not contain the answer, reply with exactly this sentence and nothing else, in English even if the question is in French: I could not find this in the course documents.

PASSAGES:
[1] (Session_1, page 2) Rome and Paris are both capitals.

[2] (Session_4, page 5) GPT-3 has 175 billion parameters.

QUESTION: How many parameters does GPT-3 have? …


In [12]:
# TODO: write `parse_citations()`.
#
#   CONTRACT
#   --------
#   parse_citations(text, n_passages) -> {"valid": [...], "invalid": [...]}
#
#   · find every bracketed number in the answer: [1], [2] …
#   · the combined form counts as several citations: "[1, 2]" and "[1,2]" are 1 and 2
#   · tolerate spaces inside the brackets: "[ 1 ]" is a citation to passage 1. Models do
#     write it that way, and a citation you silently drop is a citation you cannot check
#   · valid   = the numbers k with 1 <= k <= n_passages
#     invalid = every other number found — these are citations to passages that were
#               never given to the model, and they are hallucinated sources
#   · both lists are deduplicated and sorted ascending
#   · brackets that hold no number, like [A], are ignored entirely
#
#   Why "invalid" matters: a model that writes [7] when it was shown five passages has
#   invented a reference. You can detect that without a GPU, without a judge, and without
#   reading the answer — which makes it the cheapest quality signal in the whole pipeline.

def parse_citations(text, n_passages):
    nums = set()
    for group in re.findall(r"\[([^\[\]]*)\]", text):
        if re.fullmatch(r"\s*\d+(\s*,\s*\d+)*\s*", group):   # ignore [A] et []
            nums.update(int(x) for x in re.findall(r"\d+", group))
    valid = sorted(n for n in nums if 1 <= n <= n_passages)
    invalid = sorted(n for n in nums if not 1 <= n <= n_passages)
    return {"valid": valid, "invalid": invalid}

assert isinstance(parse_citations("[1]", 1), dict), \
    "parse_citations has no body yet — replace the `...` with your implementation."
print("ok")

ok


In [13]:
# --- ten tests ----------------------------------------------------------------
assert parse_citations("Three months [1], from the first day [2].", 3) == \
    {"valid": [1, 2], "invalid": []}, "T1: two plain citations"
assert parse_citations("as shown in [5]", 3) == \
    {"valid": [], "invalid": [5]}, "T2: 5 does not exist when there are 3 passages"
assert parse_citations("[1] and again [1]", 3) == \
    {"valid": [1], "invalid": []}, "T3: duplicates collapse"
assert parse_citations("no citations at all", 3) == \
    {"valid": [], "invalid": []}, "T4: nothing to find"
assert parse_citations("both of them [1, 2] and also [3]", 3) == \
    {"valid": [1, 2, 3], "invalid": []}, "T5: the combined form counts as several"
assert parse_citations("[0] is not a passage", 3) == \
    {"valid": [], "invalid": [0]}, "T6: numbering starts at 1, so 0 is invalid"
assert parse_citations("see section [A] of the text", 3) == \
    {"valid": [], "invalid": []}, "T7: a bracket with no number is not a citation"
assert parse_citations("[2][1]", 2) == \
    {"valid": [1, 2], "invalid": []}, "T8: the output is sorted, not in order of appearance"
# T9 — models do put spaces in there, and a dropped citation cannot be checked
assert parse_citations("as stated [ 1 ] and [2 , 3]", 3) == \
    {"valid": [1, 2, 3], "invalid": []}, "T9: spaces inside the brackets are still citations"
assert parse_citations("an empty [] bracket", 3) == \
    {"valid": [], "invalid": []}, "T9: an empty bracket is not a citation"
# T10 — the off-by-one that matters: with ONE passage, [1] is valid and [2] is not
assert parse_citations("[1]", 1)["valid"] == [1], "T10: [1] is the first passage"
assert parse_citations("[2]", 1)["invalid"] == [2], "T10: there is no second passage"

print("✅ ten tests passed")

✅ ten tests passed


In [14]:
# TODO: write `answer()` — the assistant itself.
#
#   CONTRACT
#   --------
#   answer(question, k=5, cite=True, refuse=True, history=None) -> dict
#
#   Steps: rewrite the query if there is a history (Part C does the real work — for now the
#   toolbox version returns the question unchanged), retrieve k passages, build the prompt,
#   call the model, read the citations back, resolve them to sources.
#
#   The dict has these keys:
#       "text"           the model's answer
#       "citations"      the valid citation numbers, sorted
#       "bad_citations"  the invalid ones
#       "sources"        one string per VALID citation, in the same order, shaped
#                        "Session_3 p.4"
#       "refused"        True if and only if REFUSAL appears in the answer
#       "passages"       the chunk dicts you gave the model, in the order you gave them
#       "latency_s", "tokens_out"   straight from call_model
#       "query_used"     what you actually searched with
#
#   Useful: hybrid_search(query, top_k=k) returns INDICES into CHUNK_TEXTS, best first,
#   and CHUNKS[i] is the chunk dict for index i.
#
#   ⚠️  Citation n refers to the n-th passage, and Python lists start at zero.

def rewrite_query(history, question):
    return question

def answer(question, k=5, cite=True, refuse=True, history=None):
    q = rewrite_query(history, question)
    idx = hybrid_search(q, top_k=k)
    passages = [CHUNKS[i] for i in idx]
    out = call_model(build_prompt(q, passages, cite=cite, refuse=refuse))
    cit = parse_citations(out["text"], len(passages))

    sources = [f"{passages[n - 1]['doc']} p.{passages[n - 1]['page']}" for n in cit["valid"]]
    return {"text": out["text"],
            "citations": cit["valid"],
            "bad_citations": cit["invalid"],
            "sources": sources,
            "refused": REFUSAL in out["text"],
            "passages": passages,
            "latency_s": out["latency_s"],
            "tokens_out": out["tokens_out"],
            "query_used": q}

print("ok")

ok


In [15]:
# --- the smoke test: one real question, end to end ----------------------------
r = answer(EVAL[0]["q"])

need = {"text", "citations", "bad_citations", "sources", "refused", "passages",
        "latency_s", "tokens_out", "query_used"}
assert need <= set(r), f"missing keys: {need - set(r)}"
assert isinstance(r["refused"], bool), "refused must be a boolean"
assert len(r["sources"]) == len(r["citations"]), "one source per valid citation"
assert len(r["passages"]) <= 5, "k=5 means at most five passages"

# the off-by-one, checked on real output: citation n must resolve to the n-th passage
if r["citations"]:
    c = r["citations"][0]
    p = r["passages"][c - 1]
    assert r["sources"][0] == f"{p['doc']} p.{p['page']}", \
        "citation [n] does not resolve to the n-th passage — this is the off-by-one"

print("Q:", EVAL[0]["q"])
print("\nA:", r["text"])
print("\nsources:", r["sources"], "| bad:", r["bad_citations"],
      "| refused:", r["refused"], f"| {r['latency_s']}s")

Q: Which report triggered the first AI winter?

A: The Lighthill report triggered the first AI winter [2].

sources: ['Session_1 p.4'] | bad: [] | refused: False | 8.19s


In [16]:
# Does it refuse? Ask it something the corpus cannot possibly answer.
probe = answer("Quelle est la capitale du Maroc ?")
print("A:", probe["text"][:300])
print("\nrefused:", probe["refused"])
print("\nThe model knows the answer perfectly well from pretraining. If it answered,")
print("your assistant is fluent but not grounded — note it, and come back to it in Part E.")

A: I could not find this in the course documents.

refused: True

The model knows the answer perfectly well from pretraining. If it answered,
your assistant is fluent but not grounded — note it, and come back to it in Part E.


---
## Part C — Follow-up questions (20 minutes)

Open the assistant to a real user and the second question is always *"and in which year?"*.
Sent to the retriever as it stands, that sentence matches nothing: it has no subject.

The fix is the one from Session 7, part V: **rewrite the question before you search**. The
toolbox currently defines a `rewrite_query` that returns the question unchanged — that is your
baseline. Replace it, and measure what changes.

In [26]:
# TODO: write the real `rewrite_query()`, replacing the toolbox no-op.
#
#   CONTRACT
#   --------
#   rewrite_query(history, question) -> str
#
#   history : a list of (question, answer) pairs, oldest first. May be None or empty.
#
#   · empty or None history → return the question unchanged, WITHOUT calling the model
#     (a pointless call is a pointless second of latency and a pointless token)
#   · otherwise, ask the model to rewrite the follow-up as a standalone question:
#     resolve the pronouns and the implicit references, keep the original language
#   · return ONE line: no quotes, no "Here is the rewritten question:", no explanation
#   · if what comes back is empty, or absurdly long, fall back to the original question —
#     a bad rewrite is worse than no rewrite
#
#   Use the last two or three turns, not the whole conversation: the prompt grows, the
#   latency grows, and turn fifteen rarely helps you understand turn sixteen.

def rewrite_query(history, question):
    if not history:
        return question
    convo = "\n".join(f"User: {q}\nAssistant: {a[:400]}" for q, a in history[-3:])
    prompt = ("Rewrite the user's last question as a standalone question that can be understood "
              "without the conversation: resolve pronouns and implicit references. Keep the same "
              "language as the last question. Output ONE line only: the question, with no quotes "
              "and no explanation.\n\n"
              f"Conversation:\n{convo}\n\nLast question: {question}\n\nStandalone question:")
    try:
        out = call_model(prompt, max_tokens=200)["text"]
    except Exception:
        return question
    out = out.strip().split("\n")[0].strip().strip('"').strip("'").strip()
    if not out or len(out) > 300:
        return question
    return out

assert rewrite_query(None, "What is a token?") == "What is a token?", \
    "no history → return the question unchanged"
assert rewrite_query([], "What is a token?") == "What is a token?", \
    "empty history → return the question unchanged"
print("ok")

ok


In [27]:
# Five conversations. For each: ask the first question, then ask the follow-up twice —
# once with no memory of the first turn, once with it.
FOLLOWUPS = [
 ("Who was Alan Turing?",                      "And what test is named after him?"),
 ("What is a perceptron?",                     "Why could it not solve XOR?"),
 ("What is GPT-3?",                            "How many parameters does it have?"),
 ("Qu'est-ce que l'apprentissage supervisé ?", "Et le non supervisé ?"),
 ("What is an embedding?",                     "Why is it better than one-hot vectors?"),
]

rows = []
for first, follow in FOLLOWUPS:
    r1 = answer(first)
    blind = answer(follow)                                  # no history
    aware = answer(follow, history=[(first, r1["text"])])    # with history
    ids_b = {c["chunk_id"] for c in blind["passages"]}
    ids_a = {c["chunk_id"] for c in aware["passages"]}
    rows.append({"follow-up": follow[:34],
                 "rewritten as": aware["query_used"][:46],
                 "same passages": f"{len(ids_b & ids_a)}/5",
                 "blind refused": blind["refused"],
                 "aware refused": aware["refused"]})
    print(f"  {follow[:40]:<42} → {aware['query_used'][:60]}")

followups = pd.DataFrame(rows)
print()
display(followups)

  And what test is named after him?          → What test is named after Alan Turing?
  Why could it not solve XOR?                → Why could the perceptron not solve XOR?
  How many parameters does it have?          → How many parameters does GPT-3 have?
  Et le non supervisé ?                      → Qu'est-ce que l'apprentissage non supervisé ?
  Why is it better than one-hot vectors?     → Why are embeddings better than one-hot vectors?



,follow-up,rewritten as,same passages,blind refused,aware refused
0,And what test is named after him?,What test is named after Alan Turing?,2/5,True,False
1,Why could it not solve XOR?,Why could the perceptron not solve XOR?,4/5,False,False
2,How many parameters does it have?,How many parameters does GPT-3 have?,2/5,True,False
3,Et le non supervisé ?,Qu'est-ce que l'apprentissage non supervisé ?,0/5,True,True
4,Why is it better than one-hot vect,Why are embeddings better than one-hot vectors,3/5,False,False


In [42]:
# C.1 — read your own table before moving on.
#
# The column that matters is "same passages": how many of the five retrieved chunks the two
# versions have in common. A 5/5 means the rewrite changed nothing and the question was
# already self-contained. A 0/5 or 1/5 means the rewrite sent the retriever somewhere else
# entirely — look at that row and decide whether it went somewhere BETTER.

C_OBSERVATION = """
The follow-up that changed the most is "Et le non supervisé ?" (row 3): the rewrite turned it into "Qu'est-ce que l'apprentissage non supervisé ?" and the retrieved passages share 0/5 with the blind version, but the answer did not improve, since both versions still refused. The clearest gains are rows 0 and 2: "And what test is named after him?" became "What test is named after Alan Turing?" and "How many parameters does it have?" became "How many parameters does GPT-3 have?". Both were refused blind and answered once rewritten, with 2/5 passages in common. Rows 1 and 4 changed little (4/5 and 3/5) because the question already contained its subject and neither version refused.
"""
assert len(C_OBSERVATION.strip()) > 120, "Write the observation."
print("ok")

ok


---
## Part D — The interface (15 minutes)

`launch_ui` puts your `answer()` behind a chat window and prints a public link, valid for
about 72 hours. Send it to the person next to you.

This is not decoration. You are about to find bugs that nine assert statements did not,
because a test suite asks the questions you thought of and a user asks the others.

In [29]:
launch_ui(answer)

* Running on public URL: https://71d0792401ae296627.gradio.live


In [49]:
# D.1 — use your own assistant for five minutes, then write down what you saw.
#
# Ask at least: one question you know the corpus answers, one in French, one follow-up that
# depends on the previous turn, and one you know it cannot answer.

UI_NOTES = """
I asked "What is the ELIZA effect?" and the assistant answered correctly, citing [2, 5], resolved to "Session_1 p.4, Session_1 p.2" (6.66 s in the acceptance run). The follow-up worked: after "What is GPT-3?" (answered with 175 billion parameters, 2,048-token context and 96 blocks, citing Session_4 p.3), the question "How many parameters does it have?" correctly returned 175 billion, citing Session_4 p.4 and p.3, so the history was understood. The unanswerable "Quelle est la capitale du Maroc ?" was correctly refused (7.29 s), so the assistant is grounded and not merely fluent. What broke: the French question "Qu'est-ce que l'apprentissage supervisé ?" was refused even though Session_3 defines supervised learning, the same retrieval weakness on French questions seen in Part E. A smaller issue is that raw citation numbers like [2, 5] appear in the text and mean nothing to a user, and every answer takes 6 to 8 seconds.
"""
assert len(UI_NOTES.strip()) > 150, "Write the notes."
print("ok")

ok


---
## Part E — The acceptance test (25 minutes)

You have something that answers. Now decide whether it is good enough to hand over, the way
you would decide it for any deliverable: write down what it must do, test it, and fix what
fails.

Twenty questions. **Fifteen it must answer** — your ten from Lab 5, plus five you write now —
and **five it must refuse**, because the answer is not in Sessions 1 to 4.

A question passes when:

| | an answerable question | a question it must refuse |
|---|---|---|
| **passes if** | it does not refuse · it cites at least one passage · every citation exists · at least one cited source is in the expected document | it produces the refusal sentence |

The last of the five it must refuse is *"Quelle est la capitale du Maroc ?"*. The model knows
the answer. It must refuse anyway. That single question tells you whether you built a grounded
assistant or a fluent one.

> One warning about the document check. When a question fails because the assistant cited
> Session_2 and you expected Session_3, **read the cited passage before blaming the
> assistant**. Sometimes it answered correctly from a document you had not thought of, and
> the thing that is wrong is your expected document. Correcting the test rather than the code
> is a legitimate fix — as long as you say that is what you did.

In [50]:
# TODO: write five more questions the corpus CAN answer, one per line below.
# Declare which document holds the answer. They must not duplicate your Lab 5 questions.

MY_QUESTIONS = [
    {"q": "What is the ELIZA effect?", "doc": "Session_1"},
    {"q": "Which three ingredients had to arrive together for deep learning to take off?", "doc": "Session_2"},
    {"q": "What are the training, validation and test sets each used for?", "doc": "Session_3"},
    {"q": "Why does a Transformer need positional encoding?", "doc": "Session_4"},
    {"q": "What is a base model and why does it not answer questions?", "doc": "Session_4"},
]

assert len(MY_QUESTIONS) == 5, "Five questions."
assert all(len(m["q"]) > 15 and m["q"] != "..." for m in MY_QUESTIONS), "Fill them in."
assert all(m["doc"] in {f"Session_{i}" for i in range(1, 5)} for m in MY_QUESTIONS)
assert len({m["q"] for m in MY_QUESTIONS}) == 5, "Five DIFFERENT questions."

# Each one must really be answerable: a chunk from the declared document has to show up
# in the top five. If this fails, your question is not in the corpus — rewrite it.
for m in MY_QUESTIONS:
    docs = {CHUNKS[i]["doc"] for i in hybrid_search(m["q"], top_k=5)}
    assert m["doc"] in docs, (f"{m['q']!r}: nothing from {m['doc']} in the top 5 "
                              f"(got {sorted(docs)}). Rewrite the question or fix the doc.")
print("✅ five answerable questions, all reachable")

✅ five answerable questions, all reachable


In [51]:
# The twenty questions, and the runner. Read it — you will be changing what it measures
# in a moment, and you cannot change what you have not read.

SUITE = ([dict(e, must_refuse=False) for e in EVAL] +
         [dict(m, must_refuse=False) for m in MY_QUESTIONS] +
         [{"q": u["q"], "doc": "", "must_refuse": True, "why": u["why"]} for u in UNANSWERABLE])
print(f"{len(SUITE)} questions: {sum(not s['must_refuse'] for s in SUITE)} to answer, "
      f"{sum(s['must_refuse'] for s in SUITE)} to refuse\n")

def run_acceptance(suite, k=5, **kw):
    rows = []
    for item in suite:
        r = answer(item["q"], k=k, **kw)
        cited_docs = [s.split(" p.")[0] for s in r["sources"]]
        if item["must_refuse"]:
            ok = r["refused"]
            why = "" if ok else "answered a question it cannot source"
        elif r["refused"]:
            ok, why = False, "refused an answerable question"
        elif not r["citations"]:
            ok, why = False, "answered without citing anything"
        elif r["bad_citations"]:
            ok, why = False, f"cited passages that do not exist: {r['bad_citations']}"
        elif item["doc"] and item["doc"] not in cited_docs:
            ok, why = False, f"cited {cited_docs or '—'}, expected {item['doc']}"
        else:
            ok, why = True, ""
        rows.append({"question": item["q"][:44], "must_refuse": item["must_refuse"],
                     "refused": r["refused"], "sources": ", ".join(cited_docs) or "—",
                     "pass": ok, "why": why, "s": r["latency_s"]})
    return pd.DataFrame(rows)

def report(df, label):
    a = df[~df.must_refuse]; b = df[df.must_refuse]
    print(f"{label}:  overall {df['pass'].mean():.0%}   "
          f"answerable {a['pass'].mean():.0%} ({a['pass'].sum()}/{len(a)})   "
          f"must refuse {b['pass'].mean():.0%} ({b['pass'].sum()}/{len(b)})")
    return {"overall": round(float(df['pass'].mean()), 3),
            "answerable": round(float(a['pass'].mean()), 3),
            "refusal": round(float(b['pass'].mean()), 3)}

20 questions: 15 to answer, 5 to refuse



In [52]:
accept_v1 = run_acceptance(SUITE)
score_v1 = report(accept_v1, "version 1")
display(accept_v1[~accept_v1["pass"]][["question", "must_refuse", "refused", "sources", "why"]])

version 1:  overall 90%   answerable 87% (13/15)   must refuse 100% (5/5)


,question,must_refuse,refused,sources,why
7,Why does a chatbot not remember me between t,False,True,—,refused an answerable question
8,Pourquoi l'arabe coûte-t-il plus cher que l',False,True,—,refused an answerable question


In [53]:
# TODO: pick ONE failing question. Diagnose it, change one thing, run the suite again.
#
# What you are allowed to change — one of these, not all of them:
#   · k, the number of passages retrieved
#   · the wording of your instructions in build_prompt (stronger grounding? stronger refusal?)
#   · SIZE and OVERLAP, re-chunking and re-indexing the corpus
#   · the embedding model in build_index
#
# The cache keys on the prompt, so changing any of these produces real calls again. Changing
# nothing produces the same table — which is itself a useful thing to have seen once.

DIAGNOSIS = """
Two answerable questions failed in version 1: "Why does a chatbot not remember me between two conversations?" and "Pourquoi l'arabe coûte-t-il plus cher que l'anglais pour le même texte ?". Both were refused with no source cited. Raising k from 5 to 10 changed nothing: version 2 scored exactly like version 1 (87% answerable, 100% must-refuse, zero verdict changes). A rank check showed why. The chunk holding the first answer (Session_3#16, "the weights are frozen") is not in the top 50 of either the vector or the BM25 ranking, so it is never retrieved. The chunk holding the second answer (Session_4#7) ranks 30th. In both cases the model never saw the passage and refused correctly given what it was shown. The cause is retrieval: the first question shares no words with its passage (remember versus the weights are frozen), and the French question cannot match English tokens in BM25. A larger k cannot help when the right chunk is ranked that low.
"""

WHAT_I_CHANGED = "k from 5 to 10 in run_acceptance (no effect: the right chunks rank 30th and beyond the top 50)"

# --- make your change here, then re-run -------------------------------------------
# e.g.  accept_v2 = run_acceptance(SUITE, k=8)
accept_v2 = run_acceptance(SUITE, k=10)

score_v2 = report(accept_v2, "version 2")
assert len(DIAGNOSIS.strip()) > 200, "Write the diagnosis."
assert "TODO" not in WHAT_I_CHANGED and len(WHAT_I_CHANGED.strip()) > 15, \
    "Say what you changed."

compare = accept_v1[["question", "pass"]].merge(
    accept_v2[["question", "pass"]], on="question", suffixes=("_v1", "_v2"))
changed = compare[compare.pass_v1 != compare.pass_v2]
print(f"\n{len(changed)} question(s) changed verdict:")
display(changed if len(changed) else "none — your change did nothing measurable")

version 2:  overall 90%   answerable 87% (13/15)   must refuse 100% (5/5)

0 question(s) changed verdict:


'none — your change did nothing measurable'

---
## Part F — The release note (15 minutes)

Not exam questions. What you write when you hand something over: what it does, what it
scored, what it gets wrong, and what you would do next.

Write it for the person who will inherit this and has not been in the room.

In [54]:
WHAT_IT_DOES = """
The assistant answers questions about the four course study sheets (Sessions 1 to 4). It retrieves passages with a hybrid search (multilingual embeddings plus BM25, merged by reciprocal rank fusion), sends them to Gemini at temperature 0, and answers with citations like [1] that are resolved to a document and page, for example Session_1 p.4. It works in English and French, answers in the language of the question, and replies "I could not find this in the course documents." when the passages do not hold the answer. Follow-up questions are rewritten into standalone questions before searching.
"""

MEASURED = """
On my 20-question acceptance suite (15 answerable, 5 that must be refused), version 1 scored 90% overall: 87% on answerable questions (13/15) and 100% on must-refuse questions (5/5). The two failures were refusals of answerable questions, about a chatbot not remembering the user and about Arabic costing more than English. I changed k from 5 to 10. Version 2 scored exactly the same, 87% answerable and 100% must-refuse, with zero verdicts changed, because the right chunks rank 30th and beyond the top 50. The follow-up rewrite did make a measurable difference: "How many parameters does it have?" was refused without history and answered once rewritten as "How many parameters does GPT-3 have?".
"""

KNOWN_LIMITS = """
1. Vocabulary mismatch: a question that does not reuse the words of the passage, like "remember me" versus "the weights are frozen", is never retrieved (the chunk is not in the top 50), so the assistant refuses an answerable question.
2. French questions are fragile: the Arabic-cost question ranked its chunk 30th, and "Et le non supervisé ?" was rewritten correctly yet still refused, so French against an English corpus can miss the right passage.
3. Latency: one answer took 8.19 seconds in the smoke test, and a follow-up costs an extra model call for the rewrite, which is slow for a chat interface.
4. Coverage stops at Session 4: anything from later sessions, lab details or exam dates is refused by design.
"""

NEXT_WEEK = """
I would improve retrieval for questions whose words do not match the passage: translate or expand the query before searching, for example by asking the model for a short English rewrite of French questions. Both failures I observed came from the right chunk not being retrieved, and the prompt was not at fault, since the model refused correctly what it could not see.
"""

# The question the acceptance test cannot answer for you.
BLIND_SPOT = """
My suite checks that the answer cites a passage from the expected document, not that the answer is actually true to that passage. An answer can cite the right page and still misstate a number, and my twenty questions would mark it as passed. The five refusal questions are also easy: none is a near-miss that sits close to the corpus. The suite has no Arabic question and no multi-turn conversation beyond one follow-up. I would add a check comparing each answer with its cited passage, a few near-miss unanswerable questions, and Arabic questions, since the tokenization cost in Session 4 makes Arabic a real use case in Morocco.
"""

for name, txt in [("WHAT_IT_DOES", WHAT_IT_DOES), ("MEASURED", MEASURED),
                  ("KNOWN_LIMITS", KNOWN_LIMITS), ("NEXT_WEEK", NEXT_WEEK),
                  ("BLIND_SPOT", BLIND_SPOT)]:
    assert "TODO" not in txt and len(txt.strip()) > 110, f"{name} is not written yet."
print("✅ release note complete")

✅ release note complete


In [56]:
export_lab("lab06", STUDENT_ID,
           answers={"what_it_does": WHAT_IT_DOES, "measured": MEASURED,
                    "known_limits": KNOWN_LIMITS, "next_week": NEXT_WEEK,
                    "blind_spot": BLIND_SPOT, "diagnosis": DIAGNOSIS,
                    "what_i_changed": WHAT_I_CHANGED,
                    "c_observation": C_OBSERVATION, "ui_notes": UI_NOTES},
           tables={"acceptance_v1": accept_v1, "acceptance_v2": accept_v2,
                   "followups": followups},
           extra={"size": int(SIZE), "overlap": int(OVERLAP),
                  "n_chunks": len(CHUNKS), "k": 10,
                  "score_v1": score_v1, "score_v2": score_v2,
                  "my_questions": MY_QUESTIONS,
                  "eval_from_lab5": EVAL,
                  "n_calls": len(CALL_LOG),
                  "n_cached": sum(1 for c in CALL_LOG if c.get("cached"))})

📦 lab06_130588.json written (41188 bytes)
✅ All answers filled.

Submit the notebook AND lab06_130588.json on connect.uir.ac.ma


{'lab': 'lab06',
 'student_id': '130588',
 'answers': {'what_it_does': 'The assistant answers questions about the four course study sheets (Sessions 1 to 4). It retrieves passages with a hybrid search (multilingual embeddings plus BM25, merged by reciprocal rank fusion), sends them to Gemini at temperature 0, and answers with citations like [1] that are resolved to a document and page, for example Session_1 p.4. It works in English and French, answers in the language of the question, and replies "I could not find this in the course documents." when the passages do not hold the answer. Follow-up questions are rewritten into standalone questions before searching.',
  'measured': 'On my 20-question acceptance suite (15 answerable, 5 that must be refused), version 1 scored 90% overall: 87% on answerable questions (13/15) and 100% on must-refuse questions (5/5). The two failures were refusals of answerable questions, about a chatbot not remembering the user and about Arabic costing more tha

---
### Before you close the tab

Your `llm_cache/` folder holds every answer the model gave you today. If you re-run this
notebook tomorrow with the same prompts, it costs nothing — which is what made the fix in
Part E cheap enough to be worth doing twice.

Next week the assistant stops answering and starts **acting**: it will be given tools, it
will decide which one to call, and the retrieved passage will no longer be the end of the
chain but one step in it. Keep this notebook. The `answer()` you wrote today becomes one of
those tools.